# Feature Engineering
---
*Christian Jelo R. Artoza*

Feature engineering - turning information into numbers that you can use to build your feature matrix.

## Categorical Features

### One-hot Encoding

A way to convert categorical features is to use *one-hot encoding*. That is, expanding the column containing the categorical features to a set of columns, one for each category, that indicate the presece or absence of a category with values of 1 or 0, respectively.

Scikit-Learn offers plenty of ways to do one-hot encoding including use of the `DictVectorizer()`, `OneHotEncoder()`, and `FeatureHasher()`

In [21]:
from sklearn.feature_extraction import DictVectorizer
import pandas as pd

data = [
    {'price': 850000, 'rooms': 4, 'neighborhood': 'Queen Anne'},
    {'price': 700000, 'rooms': 3, 'neighborhood': 'Fremont'},
    {'price': 650000, 'rooms': 3, 'neighborhood': 'Wallingford'},
    {'price': 600000, 'rooms': 2, 'neighborhood': 'Fremont'}
]

vec = DictVectorizer(sparse=False, dtype=int)
data_encoded = vec.fit_transform(data)

print('Original Data:')
print(pd.DataFrame(data))
print('')
print('One-Hot Encoded Data:')
print(pd.DataFrame(data_encoded, columns=vec.feature_names_))

Original Data:
    price  rooms neighborhood
0  850000      4   Queen Anne
1  700000      3      Fremont
2  650000      3  Wallingford
3  600000      2      Fremont

One-Hot Encoded Data:
   neighborhood=Fremont  neighborhood=Queen Anne  neighborhood=Wallingford  \
0                     0                        1                         0   
1                     1                        0                         0   
2                     0                        0                         1   
3                     1                        0                         0   

    price  rooms  
0  850000      4  
1  700000      3  
2  650000      3  
3  600000      2  


the above code provides a demonstration of one-hot encoding. The categorical feature 'neighborhood' is expanded to three columns: neighborhood=Fremont, neighborhood=Queen Anne, neighborhood=Wallingford. The value is 1 if the sample belongs to that neighborhood, and 0 otherwise.

### Sparsing
One disadvantage of this is that the size of the dataset becomes greatly large if you have a lot of categories. A sparsed output is a useful workaround because the columns will contain mostly zeros.

We can substitute a sparsed matrix for the original since many (but not all) Scikit-Learn estimators accept sparse inputs.

## Text Features

### Count Vectorizer
A way to process text features is by word count: take each text sample, count the occurences of each word, and put the results in a table. Scikit-Learn offers a way to do this with the `CountVectorizer()` function.

In [39]:
sample = [
    'problem of evil',
    'evil queen',
    'horizon problem'
]

from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer()
X = vec.fit_transform(sample)
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())

,evil,horizon,of,problem,queen
0,1,0,1,1,0
1,1,0,0,0,1
2,0,1,0,1,0


### Term Frequency-Inverse Document Frequency Vectorizer

The problem with having raw word count as values is that it puts too much weight on words that occur very frequently, but are not really descriptive of the text -- words like 'the', 'for', 'in', 'on' don't really help in distinguishing texts, but they are likely to occur very frequently.

A workaround for this is the *Term Frequency-Inverse Document Frequency*, which puts weight on terms based on its count and how rare it is among documents. It is composed of the following:
- Term Frequency - term count divided by the total count of words in the text
- Inverse Document Frequency - a measure of how rare a word is among the text samples; here, words like 'the' and 'of' get penalized as they occur frequently among text samples.

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

sample1 = [
    "The galaxy is full of stars and planets. The telescope captured the galaxy clearly.",
    "This recipe is simple and the recipe uses fresh tomatoes and basil.",
    "The basketball team won the championship and the basketball fans celebrated loudly."
]

# with raw word count
vec1 = CountVectorizer()
X1 = vec1.fit_transform(sample1)

# with Tfidf
vec2 = TfidfVectorizer()
X2 = vec2.fit_transform(sample1)

print('with word count:')
print(pd.DataFrame(X1.toarray(), columns=vec1.get_feature_names_out()))
print('')
print('with TF-IDF:')
print(pd.DataFrame(X2.toarray(), columns=vec2.get_feature_names_out()))

with word count:
   and  basil  basketball  captured  celebrated  championship  clearly  fans  \
0    1      0           0         1           0             0        1     0   
1    2      1           0         0           0             0        0     0   
2    1      0           2         0           1             1        0     1   

   fresh  full  ...  recipe  simple  stars  team  telescope  the  this  \
0      0     1  ...       0       0      1     0          1    3     0   
1      1     0  ...       2       1      0     0          0    1     1   
2      0     0  ...       0       0      0     1          0    3     0   

   tomatoes  uses  won  
0         0     0    0  
1         1     1    0  
2         0     0    1  

[3 rows x 25 columns]

with TF-IDF:
        and     basil  basketball  captured  celebrated  championship  \
0  0.152159  0.000000    0.000000  0.257627    0.000000      0.000000   
1  0.336500  0.284872    0.000000  0.000000    0.000000      0.000000   
2  0.1608

Observe that words that distinguish a document from the others have more weights than terms like 'the' and 'and' which occur more frequently among documents.

For example, in document 3 (index 2), the term 'basketball' has a higher weight than 'the' even though it occured less frequently.

## Image Features

- For this, we can simply use the pixels as the features like the one we did in the intro to sklearn section. This is provided that the images are given as a grid of pixels.

- ...im gonna study more about this in other books... some standard approaches can be found in *Scikit-Image project*. Definitely gonna check out the application chapter of this book as well.

## Derived Features

Derived features are those that were mathematically derived from an existing feature.

We have already encountered such feature when we used the `PolynomialFeatures()` function along side the `LinearRegression()` model in fitting higher order polynomials on training data.

naturally, the `LinearRegression()` model is used for fitting straight lines to `X,y`. But, we can also use it for fitting higher order polynomials by deriving the features `X^2`, `X^3`, and so on. 

## Imputation of Missing Data

Aside from using Pandas' `fillna()` method, Scikit-Learn offers ways to handle missing data from simple strategies like mean, median, and mode, to sophisticated strategies.

For simple strategies, we can use the `SimpleImputer` class from the `sklearn.impute` module.

In [4]:
import numpy as np
from numpy import nan
from sklearn.impute import SimpleImputer

# Example: An array with nan values
X = np.array([
    [ nan, 0, 3 ],
    [ 3, 7, 9 ],
    [ 3, 5, 2 ],
    [ 4, nan, 6 ],
    [ 8, 8, 1 ]
])

# We can use the SimpleImputer class to fill the nan values based on the strategy of choice
# Here we choose filling the nan values with the mean as our strategy
# What SimpleImputer does now is that it replaces the nan values of a feature with
# the mean of the non-nan values of that feature.
imp = SimpleImputer(strategy='mean')
X2 = imp.fit_transform(X)
X2

array([[4.5, 0. , 3. ],
       [3. , 7. , 9. ],
       [3. , 5. , 2. ],
       [4. , 5. , 6. ],
       [8. , 8. , 1. ]])

## Feature Pipelines

Scikit-Learn provides pipeline objects to streamline processing pipelines composed of steps such as pre-processing, normalization, and model fitting.

We've already encounter the use of pipelines from Course 7 and in the previous section.

For example, we have the following processing pipeline.
- Impute missing values using the mean
- Derive features for quadratic regression
- Fit the LinearRegression model to the data

Of course, we can do this one-by-one in the code, or we can do all steps in one go using Scikit-Learn.

In [7]:
import numpy as np
from numpy import nan
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline, make_pipeline

X = np.array([
    [ nan, 0, 3 ],
    [ 3, 7, 9 ],
    [ 3, 5, 2 ],
    [ 4, nan, 6 ],
    [ 8, 8, 1 ]
])
y = np.array([14, 16, -1, 8, -5])
print('Actual values:')
print(y)
print('')

# Method 1: Use the Pipeline constructor from the sklearn.pipeline module
model = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('features', PolynomialFeatures(degree=2)),
    ('fit', LinearRegression())
])

print('Using Pipeline():')
print(model.fit(X,y).predict(X))
print('')

# Method 2: Use the make_pipeline, a shorthand for the Pipeline constructor
model1 = make_pipeline(
    SimpleImputer(strategy='mean'),
    PolynomialFeatures(degree=2),
    LinearRegression()
)

print('Using make_pipeline():')
print(model1.fit(X,y).predict(X))

Actual values:
[14 16 -1  8 -5]

Using Pipeline():
[14. 16. -1.  8. -5.]

Using make_pipeline():
[14. 16. -1.  8. -5.]


One difference between the `Pipeline()` constructor and its shorthand, `make_pipeline()` is that the latter does not permit naming the steps. We have more flexibility with the `Pipeline()` constructor, but `make_pipeline()` offers convenience.

In [21]:
print('using Pipeline():')
print(model.named_steps)
print('')
print('using make_pipeline():')
print(model1.named_steps)

using Pipeline():
{'impute': SimpleImputer(), 'features': PolynomialFeatures(), 'fit': LinearRegression()}

using make_pipeline():
{'simpleimputer': SimpleImputer(), 'polynomialfeatures': PolynomialFeatures(), 'linearregression': LinearRegression()}
